# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdubakr77/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: Refresh strong content before it decays**

The paper's thesis states that useful SEO decisions came from validated comparisons, including refreshing strong content before it decays.

*Methodology question:* where does the label "decaying" come from, and does the validation design support a causal refresh recommendation? The paper defines trend direction from a 30-day-vs-previous-30-day impression change, an observational comparison, not a controlled experiment. To support the claim that refreshing content prevents decay, the study would need to show outcomes for pages that were refreshed versus similar pages that were not, not just correlate decline with staleness after the fact. As written, this reads as a directional, decision-support observation, not evidence that refreshing causes recovery.

**Finding 2: The composite Health Score is not a market-standard outcome metric**

The paper itself discloses that the Health Score (impressions 30pts + position 30pts + CTR 20pts + scroll depth 20pts) is a FlyRank summary metric, not a market-standard outcome metric, and states that major findings pair it with raw search performance rather than relying on it alone.

*Methodology question:* since the Health Score is a weighted composite of four different signals, does every finding that uses it also report the underlying raw metrics separately, or does the composite sometimes stand in for the evidence? A composite score can move even when the raw metrics tell different stories, so any claim using Health Score alone needs the raw components shown alongside it to be verifiable, exactly as the paper's own evidence standard requires.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Applying the same scrutiny to my own ML-08 work. Re-running a naive random split (what would have happened without grouping) against the honest grouped split I actually used, to show the before/after difference concretely.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Load data fresh in this notebook
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['search_volume', 'competition', 'cpc', 'word_count', 
                 'impressions_90d', 'clicks_90d', 'sessions_90d', 
                 'engaged_sessions_90d', 'scroll_events_90d',
                 'content_age_days', 'days_since_last_update',
                 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

df_clean = df.dropna(subset=feature_cols + ['label'])

# Redefine the precision@50 helper (was only in w05_model.ipynb)
def precision_at_k(df_scored, score_col, k=50):
    top_k = df_scored.sort_values(score_col, ascending=False).head(k)
    return top_k['label'].mean()

# --- BEFORE: naive random split (ignores client grouping) ---
X = df_clean[feature_cols]
y = df_clean['label']

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_naive.fit(X_train_naive, y_train_naive)

test_naive = df_clean.loc[X_test_naive.index].copy()
test_naive['model_score'] = rf_naive.predict_proba(X_test_naive)[:, 1]
naive_p50 = precision_at_k(test_naive, 'model_score', k=50)

train_clients_naive = set(df_clean.loc[X_train_naive.index, 'client_id'])
test_clients_naive = set(df_clean.loc[X_test_naive.index, 'client_id'])
overlap = train_clients_naive & test_clients_naive

# --- AFTER: grouped split by client_id (honest, matches ML-08) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_clean, groups=df_clean['client_id']))

train = df_clean.iloc[train_idx]
test = df_clean.iloc[test_idx]

X_train, y_train = train[feature_cols], train['label']
X_test, y_test = test[feature_cols], test['label']

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

test = test.copy()
test['model_score'] = rf.predict_proba(X_test)[:, 1]
model_p50 = precision_at_k(test, 'model_score', k=50)

overlap_grouped = set(train['client_id']) & set(test['client_id'])

# --- Results ---
print(f"BEFORE (naive random split): Precision@50 = {naive_p50:.3f}")
print(f"Client overlap in naive split: {len(overlap)} clients appear in both train and test")
print(f"\nAFTER (grouped split): Precision@50 = {model_p50:.3f}")
print(f"Client overlap in grouped split: {len(overlap_grouped)} clients")

BEFORE (naive random split): Precision@50 = 0.960
Client overlap in naive split: 27 clients appear in both train and test

AFTER (grouped split): Precision@50 = 0.800
Client overlap in grouped split: 0 clients


**Result:** the naive random split reported an inflated Precision@50 of 0.960, driven by 27 clients appearing in both train and test, the model was partly recognizing client-specific patterns it had already seen, not generalizing. The honest grouped split reports 0.800 with zero client overlap, a lower number, but the only one that reflects how the model would perform on a genuinely new client. The 0.160 gap between these two numbers is the size of the leakage risk that a casual random split would have hidden.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Reviewed all 15 features used in the model against the label definition (`trend_direction == 'down'`):

- `trend_pct` was excluded from the feature set entirely, it is derived from the same impression comparison that defines the label itself; including it would have been direct label leakage.
- `days_since_last_update`, `impressions_90d`, and the other 13 features are all observable independently of the outcome window used to compute `trend_direction`, they describe the page's state and history, not its future trend.
- No client-identifying fields (`client_id`, `content_id`) were used as model features, only as grouping keys for the split, so no client identity leaked into the model's predictions.

**One limitation found during this audit:** `content_age_days`, the single most important feature per permutation importance in ML-08, was not explicitly checked for correlation with the 30-day window used to define `trend_direction`. Very new content could mechanically show inflated trend percentages simply due to a low prior baseline. This needs a follow-up check before treating `content_age_days` as a fully clean signal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (from ML-07 baseline write-up):** "a fixed rule like 'flag pages older than 180 days with high traffic' only uses two signals at a fixed threshold... the real pattern depends on several signals interacting together."

**Revised (safe language):** "On this sample, a rule combining staleness and visibility performed less consistently than a model considering multiple signals together (Precision@50 of 0.520 versus 0.800, measured on a client-grouped split). This is an observed, decision-support result on one dataset and one split, not a general claim that hand-written rules always underperform learned models."

The original version implied a universal truth about rules versus models. The revision scopes the claim to what was actually measured: this specific sample, this specific split, this specific metric, matching the paper's own evidence standard of pairing composite claims with the raw numbers behind them.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.